# Colab GPU Bridge: Ollama + authenticated static ngrok tunnel

Runs Ollama on Colab's free T4 GPU and exposes it over a **fixed** HTTPS URL, so your phone/VS Code client never needs reconfiguring after a Colab restart.

Ollama has no built-in authentication, so the tunnel does **not** point at it directly. A small reverse proxy sits in front, requires a bearer token on every request, and only forwards inference endpoints -- model-management routes like `/api/delete` and `/api/pull` are not reachable from the internet.

## Before you run this
1. **Runtime > Change runtime type > T4 GPU**, then Connect.
2. Create a free ngrok account: https://dashboard.ngrok.com/signup
3. Copy your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken
4. Reserve one free static domain: https://dashboard.ngrok.com/domains -> Create Domain.
5. Open the Secrets panel (key icon, left sidebar) and add two secrets, both with notebook access enabled:
   - `NGROK_AUTHTOKEN` -- the token from step 3.
   - `BRIDGE_TOKEN` -- a password you invent for your own clients. Make it long and random; you can generate one with `python -c "import secrets; print(secrets.token_urlsafe(32))"`. Storing it as a secret keeps it stable across restarts, so your clients only get configured once.
6. Fill in `NGROK_DOMAIN` and `MODEL` in the CONFIG cell below.
7. Runtime > Run all. On every later session, just Run all again -- the URL and token both stay the same.

In [ ]:
# CONFIG -- edit these two lines
NGROK_DOMAIN = "your-reserved-domain.ngrok-free.app"  # from https://dashboard.ngrok.com/domains
MODEL = "llama3.1:8b"  # 8B fits the T4's 16GB VRAM comfortably; try "qwen2.5:14b-instruct-q4_K_M" for a bigger quantized model

PROXY_PORT = 8000  # the authenticated proxy; this is what gets tunnelled, NOT Ollama's 11434

!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
# Start the Ollama server in the background, then pull the model.
# It binds to 127.0.0.1 only -- nothing reaches it except the proxy below.
import subprocess, time, requests

ollama_proc = subprocess.Popen(
    ["ollama", "serve"],
    stdout=open("/content/ollama.log", "w"),
    stderr=subprocess.STDOUT,
)

for _ in range(30):
    try:
        requests.get("http://127.0.0.1:11434")
        break
    except requests.exceptions.ConnectionError:
        time.sleep(1)
else:
    raise RuntimeError("Ollama server did not come up -- check /content/ollama.log")

print("Ollama is up, pulling model (this can take a few minutes)...")
!ollama pull {MODEL}

## Authenticating proxy

Everything past this point is what makes the tunnel safe to leave running. The proxy:

- rejects any request without `Authorization: Bearer $BRIDGE_TOKEN` (compared in constant time),
- forwards only inference and read-only routes, so `/api/pull`, `/api/push`, `/api/delete`, `/api/create` and `/api/copy` return 403 even with a valid token,
- streams responses through chunk-by-chunk, so token-by-token output still arrives live.

In [ ]:
!pip install -q aiohttp

import asyncio, secrets, threading
from aiohttp import web, ClientSession, ClientTimeout
from google.colab import userdata

try:
    BRIDGE_TOKEN = userdata.get("BRIDGE_TOKEN")
except Exception:
    BRIDGE_TOKEN = None

if not BRIDGE_TOKEN:
    BRIDGE_TOKEN = secrets.token_urlsafe(32)
    print("!! No BRIDGE_TOKEN secret found, generated a temporary one:\n")
    print(f"     {BRIDGE_TOKEN}\n")
    print("   This changes every restart, so you would have to reconfigure every client each time.")
    print("   Save it as a Colab secret named BRIDGE_TOKEN to make it stable.")

OLLAMA = "http://127.0.0.1:11434"

# Inference + read-only routes only. Model management is deliberately absent.
ALLOWED_PREFIXES = (
    "/v1/chat/completions", "/v1/completions", "/v1/models", "/v1/embeddings",
    "/api/chat", "/api/generate", "/api/embed", "/api/embeddings",
    "/api/tags", "/api/show", "/api/ps",
)
HOP_BY_HOP = {"content-length", "transfer-encoding", "content-encoding", "connection"}


def _token_of(request):
    auth = request.headers.get("Authorization", "")
    if auth.startswith("Bearer "):
        return auth[len("Bearer "):]
    return request.headers.get("X-Api-Key", "")


async def handler(request):
    if not secrets.compare_digest(_token_of(request), BRIDGE_TOKEN):
        return web.json_response({"error": "unauthorized"}, status=401)

    if not any(request.path.startswith(p) for p in ALLOWED_PREFIXES):
        return web.json_response(
            {"error": f"{request.path} is not exposed by this bridge"}, status=403
        )

    target = OLLAMA + request.path_qs
    fwd_headers = {
        k: v for k, v in request.headers.items()
        if k.lower() not in HOP_BY_HOP | {"host", "authorization", "x-api-key"}
    }
    body = await request.read()

    async with ClientSession(timeout=ClientTimeout(total=None)) as session:
        async with session.request(
            request.method, target, data=body, headers=fwd_headers
        ) as upstream:
            resp = web.StreamResponse(
                status=upstream.status,
                headers={
                    k: v for k, v in upstream.headers.items()
                    if k.lower() not in HOP_BY_HOP
                },
            )
            await resp.prepare(request)
            async for chunk in upstream.content.iter_any():
                await resp.write(chunk)
            await resp.write_eof()
            return resp


app = web.Application(client_max_size=64 * 1024 * 1024)
app.router.add_route("*", "/{tail:.*}", handler)


def _serve():
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    runner = web.AppRunner(app)
    loop.run_until_complete(runner.setup())
    loop.run_until_complete(web.TCPSite(runner, "127.0.0.1", PROXY_PORT).start())
    loop.run_forever()


threading.Thread(target=_serve, daemon=True).start()
time.sleep(2)

# Sanity check: no token must fail, correct token must succeed.
import requests

anon = requests.get(f"http://127.0.0.1:{PROXY_PORT}/api/tags")
authed = requests.get(
    f"http://127.0.0.1:{PROXY_PORT}/api/tags",
    headers={"Authorization": f"Bearer {BRIDGE_TOKEN}"},
)
blocked = requests.post(
    f"http://127.0.0.1:{PROXY_PORT}/api/delete",
    headers={"Authorization": f"Bearer {BRIDGE_TOKEN}"},
    json={"name": MODEL},
)

assert anon.status_code == 401, f"expected 401 without a token, got {anon.status_code}"
assert authed.status_code == 200, f"expected 200 with a token, got {authed.status_code}"
assert blocked.status_code == 403, f"expected 403 for /api/delete, got {blocked.status_code}"
print(f"Proxy healthy on 127.0.0.1:{PROXY_PORT} (401 without token, 200 with, 403 on management routes)")

In [ ]:
# Install ngrok, authenticate from the Secrets panel, and tunnel the PROXY (not Ollama)
from google.colab import userdata

NGROK_AUTHTOKEN = userdata.get("NGROK_AUTHTOKEN")

!curl -sSL https://ngrok-agent.s3.amazonaws.com/ngrok.asc | tee /etc/apt/trusted.gpg.d/ngrok.asc >/dev/null
!echo "deb https://ngrok-agent.s3.amazonaws.com buster main" | tee /etc/apt/sources.list.d/ngrok.list
!apt update -qq && apt install -y ngrok -qq

!ngrok config add-authtoken {NGROK_AUTHTOKEN}

import subprocess, time, requests

ngrok_proc = subprocess.Popen(
    ["ngrok", "http", str(PROXY_PORT), "--domain", NGROK_DOMAIN, "--log", "stdout"],
    stdout=open("/content/ngrok.log", "w"),
    stderr=subprocess.STDOUT,
)

time.sleep(5)
url = f"https://{NGROK_DOMAIN}"

try:
    probe = requests.get(
        f"{url}/api/tags",
        headers={"Authorization": f"Bearer {BRIDGE_TOKEN}"},
        timeout=15,
    )
    if probe.status_code == 200:
        print(f"Bridge is live: {url}")
        print(f"Models visible: {[m['name'] for m in probe.json().get('models', [])]}")
    else:
        print(f"Tunnel reachable but returned {probe.status_code}: {probe.text[:200]}")
except requests.exceptions.RequestException as e:
    print(f"Tunnel not responding yet, check /content/ngrok.log -- {e}")

print("\nConfigure your client with:")
print(f"  Base URL : {url}/v1     (OpenAI-compatible mode)")
print(f"  API key  : your BRIDGE_TOKEN")
print(f"  Model    : {MODEL}")

## Optional: keep the session alive a little longer

Colab still disconnects idle free-tier sessions on its own schedule regardless of this cell -- this only stops *your* notebook from looking idle due to zero cell activity. Keep the interval modest; don't turn this into a tight polling loop, that's against Colab's free-tier usage terms.

In [ ]:
import time, datetime

try:
    while True:
        print(f"[{datetime.datetime.now().isoformat(timespec='seconds')}] bridge alive at https://{NGROK_DOMAIN}")
        time.sleep(300)
except KeyboardInterrupt:
    print("Stopped heartbeat -- Ollama, the proxy and ngrok are still running in the background.")